# auditpaper-standards MCP 서버 — Colab 호스팅

한국 회계감사기준·K-IFRS·실무지침 검색 MCP 서버를 Colab에서 HTTP로 띄우고,
터널 URL을 통해 다른 컴퓨터의 Claude Code가 붙을 수 있게 한다.
(상시 호스팅이 필요하면 HuggingFace Space 경로를 권장 — 저장소 `deploy/hf_space/` 참조.)

## 사전 준비 (한 번만)

왼쪽 사이드바 🔑 **보안 비밀(Secrets)** 에 아래 값을 등록하고 "노트북 액세스"를 켠다:

| 이름 | 값 | 필수 |
|---|---|---|
| `QDRANT_URL` | Qdrant Cloud 클러스터 URL | ✅ |
| `QDRANT_API_KEY` | Qdrant API 키 | ✅ |
| `MCP_AUTH_TOKEN` | 직접 정하는 접속 비밀번호 (**16자 이상**, 예: `openssl rand -hex 24` 출력) | ✅ |
| `NGROK_AUTHTOKEN` | [ngrok 대시보드](https://dashboard.ngrok.com)의 Authtoken (무료 계정) | 선택 |
| `NGROK_DOMAIN` | ngrok 무료 고정 도메인 (예: `xxxx.ngrok-free.app`) — 있으면 URL이 세션마다 안 바뀜 | 선택 |

터널: `NGROK_AUTHTOKEN`이 있으면 ngrok(고정 도메인 가능), 없으면 cloudflared 임시
터널(무계정·URL은 세션마다 변경). 어느 쪽이든 ⑤셀이 개통 후 라우팅을 실검사한다.
(참고: 과거 이 노트북이 겪은 421 오류는 터널이 아니라 서버의 Host 헤더 가드가
원인이었고 서버 쪽에서 수정됨 — `docs/LEDGER.md` D-21.)

## 실행

런타임은 **CPU면 충분** (GPU가 잡히면 인코딩이 더 빨라질 뿐). 셀을 위에서 아래로 전부 실행하면
마지막에 상대 컴퓨터에 전달할 `.mcp.json` 설정이 출력된다.

In [ ]:
# ① 저장소 클론 + 의존성 설치 (~2분)
![ -d /content/auditPaper_assist ] || git clone --depth 1 https://github.com/worldoftoddl/auditPaper_assist.git /content/auditPaper_assist
%pip install -q fastmcp==3.4.3 qdrant-client==1.18.0 kiwipiepy==0.23.2 "sentence-transformers>=5,<6" pyngrok
print("설치 완료")

In [ ]:
# ② (선택) bge-m3 모델 캐시를 Google Drive에 보존 — 세션마다 2.3GB 재다운로드 방지
USE_DRIVE_CACHE = True  # 원치 않으면 False

import os
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    print("HF 캐시 →", os.environ["HF_HOME"])
else:
    print("세션 로컬 캐시 사용 (세션마다 모델 재다운로드)")

In [ ]:
# ③ 비밀값 로드 (Colab Secrets → 없으면 입력 프롬프트)
import os
from getpass import getpass

def secret(name, required=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
    except Exception:
        v = None
    if not v and required:
        v = getpass(f"{name}: ")
    return v or None

os.environ["QDRANT_URL"] = secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = secret("QDRANT_API_KEY")
AUTH_TOKEN = secret("MCP_AUTH_TOKEN")
assert len(AUTH_TOKEN) >= 16, "MCP_AUTH_TOKEN은 16자 이상이어야 함 (서버가 기동 거부)"
NGROK_AUTHTOKEN = secret("NGROK_AUTHTOKEN", required=False)
NGROK_DOMAIN = secret("NGROK_DOMAIN", required=False)
print("비밀값 준비 완료 — 터널:", "ngrok" if NGROK_AUTHTOKEN else "cloudflared(무계정·임시 URL)")

In [ ]:
# ④ MCP 서버 기동 (HTTP + Bearer 인증). 기동 검증 실패 시 로그와 함께 중단.
import subprocess, sys, time, pathlib

PORT = 8765
LOG = "/content/mcp_server.log"
env = dict(os.environ, MCP_TRANSPORT="http", MCP_HOST="127.0.0.1", MCP_PORT=str(PORT),
           MCP_AUTH_TOKEN=AUTH_TOKEN, PYTHONPATH="/content/auditPaper_assist")
server = subprocess.Popen([sys.executable, "-m", "server.mcp_server"],
                          cwd="/content/auditPaper_assist", env=env,
                          stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

for _ in range(120):
    log = pathlib.Path(LOG).read_text(errors="ignore")
    if "기동 (http" in log:
        print("서버 기동 완료 — 인코더는 백그라운드 로드 중 (search는 잠시 후 활성)")
        break
    if server.poll() is not None:
        print(log)
        raise RuntimeError("서버 기동 실패 — 위 로그의 [기동 거부] 사유 확인")
    time.sleep(2)
else:
    print(pathlib.Path(LOG).read_text(errors="ignore"))
    raise RuntimeError("기동 대기 시간 초과")

# 인증 게이트 확인: 무토큰 401, 정토큰 200
import requests
init = {"jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {"protocolVersion": "2024-11-05", "capabilities": {},
                   "clientInfo": {"name": "colab-smoke", "version": "0"}}}
hdr = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
no_auth = requests.post(f"http://127.0.0.1:{PORT}/mcp", json=init, headers=hdr).status_code
ok = requests.post(f"http://127.0.0.1:{PORT}/mcp", json=init,
                   headers={**hdr, "Authorization": f"Bearer {AUTH_TOKEN}"}).status_code
assert no_auth == 401 and ok == 200, f"인증 게이트 이상: 무토큰 {no_auth}, 정토큰 {ok}"
print(f"인증 게이트 정상 (무토큰 {no_auth} / 정토큰 {ok})")

In [ ]:
# ⑤ 터널 개통 + 라우팅 검증 → 상대 컴퓨터에 전달할 설정 출력
import json, re, subprocess, time, pathlib, requests

def routed_ok(url, tries=9, wait=5):
    """터널이 서버까지 실제로 라우팅되는지 확인 — 무토큰 GET /mcp는 401이 정답.
    421이면 서버 코드가 구버전(Host 가드 미개방 — D-21), 000이면 터널 미개통."""
    for _ in range(tries):
        try:
            if requests.get(f"{url}/mcp", timeout=10).status_code == 401:
                return True
        except requests.RequestException:
            pass
        time.sleep(wait)
    return False

if NGROK_AUTHTOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    kw = {"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}
    public_url = ngrok.connect(PORT, "http", **kw).public_url
    assert routed_ok(public_url), "ngrok 터널 라우팅 실패 — 셀을 다시 실행해 볼 것"
else:
    # cloudflared 임시 터널 (무계정·URL 세션마다 변경). 개통 실패 대비 최대 2회 재생성.
    bin_path = "/content/cloudflared"
    if not pathlib.Path(bin_path).exists():
        subprocess.run(["wget", "-q", "-O", bin_path,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                       check=True)
        subprocess.run(["chmod", "+x", bin_path], check=True)
    public_url = None
    for attempt in (1, 2):
        tlog = f"/content/cloudflared_{attempt}.log"
        cf = subprocess.Popen([bin_path, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                               "--no-autoupdate"],
                              stdout=open(tlog, "w"), stderr=subprocess.STDOUT)
        url = None
        for _ in range(30):
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                          pathlib.Path(tlog).read_text(errors="ignore"))
            if m:
                url = m.group(0)
                break
            time.sleep(2)
        if url:
            print(f"시도 {attempt}: {url} — 라우팅 검사 중(최대 45초)…")
            if routed_ok(url):
                public_url = url
                break
        cf.kill()
        print(f"시도 {attempt}: 터널 라우팅 실패 — 재생성")
    assert public_url, (
        "터널 라우팅 실패. 421이 반복되면 ①셀 clone이 구버전일 수 있음 — "
        "/content/auditPaper_assist 삭제 후 ①부터 재실행 (docs/LEDGER.md D-21). "
        "000/타임아웃이면 ngrok(NGROK_AUTHTOKEN 등록) 사용 권장")

print("공개 URL 라우팅 정상:", public_url)

mcp_config = {"mcpServers": {"auditpaper-standards": {
    "type": "http",
    "url": f"{public_url}/mcp",
    "headers": {"Authorization": f"Bearer {AUTH_TOKEN}"}}}}

print("\n상대 컴퓨터의 프로젝트 루트 .mcp.json에 넣을 내용")
print("(⚠️ 접속 토큰 포함 — 이 출력을 공개 장소에 붙여넣지 말 것):\n")
print(json.dumps(mcp_config, indent=2, ensure_ascii=False))

In [ ]:
# ⑥ 원격 스모크 테스트 — 공개 URL을 실제 MCP 클라이언트로 왕복 확인
import asyncio
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

async def smoke():
    t = StreamableHttpTransport(f"{public_url}/mcp",
                                headers={"Authorization": f"Bearer {AUTH_TOKEN}"})
    async with Client(t) as c:
        print("도구:", [x.name for x in await c.list_tools()])
        r = await c.call_tool("standards_get_paragraph", {"cid": "KIFRS::1115::31"})
        p = r.data["paragraphs"][0]
        print(f"조회 확인: {p['cid']} — {p['standard_title']} (컬렉션 {r.data['collection']})")

await smoke()
print("\n✅ 원격 접속 정상 — 위 ⑤의 설정을 상대 작업자에게 전달하면 됨")

## 운영 노트

- **터널 선택**: cloudflared(무계정)와 ngrok 모두 동작한다. ngrok 고정 도메인
  (`NGROK_DOMAIN`)을 쓰면 재기동해도 URL이 같아서 상대 설정을 다시 바꿀 필요가 없다 —
  이게 ngrok을 쓸 실질적 이유. 고정 도메인 없는 ngrok·cloudflared는 세션마다 URL이 바뀐다.
- **세션 수명**: Colab Pro는 최대 24시간이며, 탭을 닫거나 유휴가 길어지면 세션이 끊긴다
  (백그라운드 실행은 Pro+ 전용). 끊기면 ①부터 재실행 — Drive 캐시(②)를 켜뒀으면 모델
  재다운로드는 건너뛴다. 상시 가동이 필요하면 HF Space(`deploy/hf_space/`)로.
- **검색 지연**: 기동 직후 인코더(bge-m3) 로드 중에는 `standards_search`만 대기가 걸릴 수
  있다 (`standards_get_paragraph`·`standards_define_terms`는 즉시 동작). 서버 로그는
  `/content/mcp_server.log`.
- **421이 다시 보이면**: 서버 코드가 구버전(Host 가드 미개방)일 가능성 — ①셀의 clone이
  최신 main인지 확인하고 `/content/auditPaper_assist`를 지운 뒤 ①부터 재실행 (D-21).
- **토큰 교체**: 토큰이 유출된 것 같으면 Secrets의 `MCP_AUTH_TOKEN`을 바꾸고 ③~⑤를
  재실행한 뒤 상대에게 새 설정을 전달한다.
- **연결 확인(상대 컴퓨터)**: `.mcp.json` 저장 → Claude Code 재시작 → 프로젝트 MCP 승인 →
  `claude mcp list`에서 ✔ Connected 확인.